# Import Data & Functions

In [1]:
import pandas as pd
import numpy as np

# CSV 파일 읽기
df = pd.read_csv('https://raw.githubusercontent.com/sw1kwon/usa-elections/refs/heads/main/clean/Presidential_Elections/1976-2020-president_v1.csv')

In [2]:
df.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,candidate,party_detailed,writein,candidatevotes,totalvotes,version,notes,party_simplified
0,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"CARTER, JIMMY",DEMOCRAT,False,659170,1182850,20210113,NaN,DEMOCRAT
1,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"FORD, GERALD",REPUBLICAN,False,504070,1182850,20210113,NaN,REPUBLICAN
2,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"MADDOX, LESTER",AMERICAN INDEPENDENT PARTY,False,9198,1182850,20210113,NaN,OTHER
3,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"BUBAR, BENJAMIN """"BEN""""",PROHIBITION,False,6669,1182850,20210113,NaN,OTHER
4,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"HALL, GUS",COMMUNIST PARTY USE,False,1954,1182850,20210113,NaN,OTHER


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4287 entries, 0 to 4286
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              4287 non-null   int64  
 1   state             4287 non-null   object 
 2   state_po          4287 non-null   object 
 3   state_fips        4287 non-null   int64  
 4   state_cen         4287 non-null   int64  
 5   state_ic          4287 non-null   int64  
 6   office            4287 non-null   object 
 7   candidate         4000 non-null   object 
 8   party_detailed    3831 non-null   object 
 9   writein           4287 non-null   bool   
 10  candidatevotes    4287 non-null   int64  
 11  totalvotes        4287 non-null   int64  
 12  version           4287 non-null   int64  
 13  notes             0 non-null      float64
 14  party_simplified  4287 non-null   object 
dtypes: bool(1), float64(1), int64(7), object(6)
memory usage: 473.2+ KB


In [4]:
df['party_simplified'].value_counts()

,count
party_simplified,
OTHER,2524
DEMOCRAT,615
REPUBLICAN,613
LIBERTARIAN,535


In [5]:
import pandas as pd
import numpy as np


def analyze_vote_aggregation(df, year):
    """
    정당별 득표수를 집계하여 미국 대통령 선거 데이터를 분석하는 함수

    Parameters:
    -----------
    df : pandas.DataFrame
        미국 선거 데이터 (1976-2020)
    year : int
        분석할 선거 연도 (1976, 1980, 1984, 1988, 1992, 1996, 2000, 2004, 2008, 2012, 2016, 2020)

    Returns:
    --------
    pandas.DataFrame
        주별 득표수 집계 분석 결과
    """

    # 연도 유효성 검사
    valid_years = list(range(1976, 2021, 4))

    if year not in valid_years:
        raise ValueError(f"유효하지 않은 연도입니다. 가능한 연도: {valid_years}")

    # 해당 연도 데이터가 존재하는지 확인
    if year not in df['year'].values:
        raise ValueError(f"{year}년 데이터가 데이터셋에 존재하지 않습니다.")

    # 제외할 candidate 값들 정의
    exclude_candidates = ['BLANK VOTE', 'OVER VOTE', 'OVERVOTES', 'UNDERVOTES', 'VOID', 'VOID VOTE']

    # 메서드 체이닝으로 데이터 처리
    result = (df
        # 지정된 연도 데이터만 필터링
        .query(f'year == {year}')

        # 제외할 candidate 행들 필터링
        .query('candidate not in @exclude_candidates')

        # writein 컬럼의 NA 값 처리
        .assign(writein=lambda x: x['writein'].fillna(True))

        # 주별로 그룹화하여 득표수 집계
        .groupby('state')
        .apply(lambda group: pd.Series({
            'election_year': group['year'].iloc[0],
            'state_code': group['state_po'].iloc[0],

            # total_votes를 group의 candidatevotes 합계로 계산
            'total_votes': group['candidatevotes'].sum(),

            # NY 주의 경우 특별 처리
            'votes_republican': (
                group.query('writein == False & party_simplified == "REPUBLICAN"')['candidatevotes'].sum()
                if group['state_po'].iloc[0] != 'NY'
                else (
                    group.query('writein == False & party_simplified == "REPUBLICAN"')['candidatevotes'].sum() +
                    group[
                        group['candidate'].isin(
                            group.query('party_simplified == "REPUBLICAN"')['candidate'].unique()
                        )
                    ]['candidatevotes'].sum()
                )
            ),

            'votes_democrat': (
                group.query('writein == False & party_simplified == "DEMOCRAT"')['candidatevotes'].sum()
                if group['state_po'].iloc[0] != 'NY'
                else (
                    group.query('writein == False & party_simplified == "DEMOCRAT"')['candidatevotes'].sum() +
                    group[
                        group['candidate'].isin(
                            group.query('party_simplified == "DEMOCRAT"')['candidate'].unique()
                        )
                    ]['candidatevotes'].sum()
                )
            ),

            'votes_libertarian': (
                group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidatevotes'].sum()
                if group['state_po'].iloc[0] != 'NY'
                else (
                    group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidatevotes'].sum() +
                    group[
                        group['candidate'].isin(
                            group.query('party_simplified == "LIBERTARIAN"')['candidate'].unique()
                        )
                    ]['candidatevotes'].sum()
                )
            ),

            'votes_other': (
                group.query('writein == False & party_simplified == "OTHER"')['candidatevotes'].sum()
                if group['state_po'].iloc[0] != 'NY'
                else (
                    group.query('writein == False & party_simplified == "OTHER"')['candidatevotes'].sum() -
                    # REPUBLICAN, DEMOCRAT, LIBERTARIAN 후보가 OTHER로 분류된 득표수 빼기
                    group[
                        (group['party_simplified'] == 'OTHER') &
                        (group['candidate'].isin(
                            group.query('party_simplified == "REPUBLICAN"')['candidate'].unique()
                        ) |
                        group['candidate'].isin(
                            group.query('party_simplified == "DEMOCRAT"')['candidate'].unique()
                        ) |
                        group['candidate'].isin(
                            group.query('party_simplified == "LIBERTARIAN"')['candidate'].unique()
                        ))
                    ]['candidatevotes'].sum()
                )
            ),

            # 모든 기명투표 득표수 합계
            'votes_writein': group.query('writein == True')['candidatevotes'].sum()
        }), include_groups=False)

        # 인덱스를 컬럼으로 변환
        .reset_index()

        # 주 이름 기준으로 정렬
        .sort_values('state')

        # 컬럼 선택 및 순서 정렬
        [['election_year', 'state', 'state_code', 'total_votes', 'votes_republican',
          'votes_democrat', 'votes_libertarian', 'votes_other', 'votes_writein']]

        # 인덱스 재설정
        .reset_index(drop=True)

        .rename(columns={
        "election_year": "year",
        "state_code": "state_po",
        "total_votes": "totalvotes"
        })
    )

    return result

def process_all_years_vote_aggregation(df, years=None):
    """
    여러 연도의 득표수 집계를 처리하는 함수

    Parameters:
    -----------
    df : pandas.DataFrame
        미국 선거 데이터
    years : list, optional
        처리할 연도 리스트. None이면 모든 가능한 연도를 처리

    Returns:
    --------
    pandas.DataFrame
        모든 연도의 결과를 합친 데이터프레임
    """

    if years is None:
        # 데이터셋에 존재하는 유효한 연도들 가져오기
        valid_years = list(range(1976, 2021, 4))
        existing_years = df['year'].unique()
        years = [year for year in valid_years if year in existing_years]

    # 각 연도별로 처리
    results = []
    for year in years:
        try:
            result = analyze_vote_aggregation(df, year)
            results.append(result)
            print(f"{year}년 분석 완료: {len(result)}개 주/지역")
        except Exception as e:
            print(f"{year}년 분석 실패: {e}")

    # 모든 결과 합치기
    if results:
        combined_result = pd.concat(results, ignore_index=True)
        return combined_result
    else:
        return pd.DataFrame()

### 사용 예시
# result_1976 = analyze_vote_aggregation(df, 1976)
# result_multi = process_all_years_vote_aggregation(df, [1976, 1980, 1984])
# result_all = process_all_years_vote_aggregation(df)  # 모든 연도

In [6]:
import pandas as pd
import numpy as np

def analyze_election_data(df, year):
    """
    미국 대통령 선거 데이터를 분석하는 함수

    Parameters:
    -----------
    df : pandas.DataFrame
        미국 선거 데이터 (1976-2020)
    year : int
        분석할 선거 연도

    Returns:
    --------
    pandas.DataFrame
        주별 1위, 2위 정당 및 득표율 정보
    """

    # 연도 유효성 검사
    valid_years = list(range(1976, 2021, 4))

    if year not in valid_years:
        raise ValueError(f"유효하지 않은 연도입니다. 가능한 연도: {valid_years}")

    # 해당 연도 데이터가 존재하는지 확인
    if year not in df['year'].values:
        raise ValueError(f"{year}년 데이터가 데이터셋에 존재하지 않습니다.")

    # 제외할 candidate 값들 정의
    exclude_candidates = ['BLANK VOTE', 'OVER VOTE', 'OVERVOTES', 'UNDERVOTES', 'VOID', 'VOID VOTE']

    # 필터링된 데이터 준비
    filtered_df = (df
        .query(f'year == {year}')
        .query('candidate not in @exclude_candidates')
        .assign(writein=lambda x: x['writein'].fillna(True))
    )

    # 주별로 그룹화하여 분석
    result = (filtered_df
        .groupby('state')
        .apply(lambda group: pd.Series({
            'year': group['year'].iloc[0],
            'state_po': group['state_po'].iloc[0],

            # NY 주와 그 외 주 구분
            'first_place_party': (
                group.nlargest(1, 'candidatevotes')['party_simplified'].iloc[0]
                if group['state_po'].iloc[0] != 'NY'
                else (lambda: (
                    # NY 주: 주요 3개 정당 득표 합산
                    votes_dict := {
                        'REPUBLICAN': (
                            group.query('writein == False & party_simplified == "REPUBLICAN"')['candidatevotes'].sum() +
                            group[
                                (group['writein'] == False) &
                                (group['party_simplified'] != 'REPUBLICAN') &
                                (group['candidate'].isin(
                                    group.query('writein == False & party_simplified == "REPUBLICAN"')['candidate'].unique()
                                ))
                            ]['candidatevotes'].sum()
                        ),
                        'DEMOCRAT': (
                            group.query('writein == False & party_simplified == "DEMOCRAT"')['candidatevotes'].sum() +
                            group[
                                (group['writein'] == False) &
                                (group['party_simplified'] != 'DEMOCRAT') &
                                (group['candidate'].isin(
                                    group.query('writein == False & party_simplified == "DEMOCRAT"')['candidate'].unique()
                                ))
                            ]['candidatevotes'].sum()
                        ),
                        'LIBERTARIAN': (
                            group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidatevotes'].sum() +
                            group[
                                (group['writein'] == False) &
                                (group['party_simplified'] != 'LIBERTARIAN') &
                                (group['candidate'].isin(
                                    group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidate'].unique()
                                ))
                            ]['candidatevotes'].sum()
                        )
                    },
                    # OTHER 후보들 개별로 추가
                    other_candidates := group[
                        (group['writein'] == False) &
                        (group['party_simplified'] == 'OTHER') &
                        (~group['candidate'].isin(
                            group.query('writein == False & party_simplified == "REPUBLICAN"')['candidate'].unique()
                        )) &
                        (~group['candidate'].isin(
                            group.query('writein == False & party_simplified == "DEMOCRAT"')['candidate'].unique()
                        )) &
                        (~group['candidate'].isin(
                            group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidate'].unique()
                        ))
                    ],
                    # OTHER 후보들을 딕셔너리에 추가
                    [votes_dict.update({f"OTHER_{row['candidate']}": row['candidatevotes']})
                     for _, row in other_candidates.iterrows()],
                    # 최대 득표 정당/후보 찾기
                    max_key := max(votes_dict, key=votes_dict.get),
                    # OTHER_{candidate} 형식이면 OTHER로 변환
                    'OTHER' if max_key.startswith('OTHER_') else max_key
                )[-1])()
            ),

            'first_place_vote_share': (
                round(
                    group.nlargest(1, 'candidatevotes')['candidatevotes'].iloc[0] /
                    group['candidatevotes'].sum(), 4
                )
                if group['state_po'].iloc[0] != 'NY'
                else (lambda: (
                    votes_dict := {
                        'REPUBLICAN': (
                            group.query('writein == False & party_simplified == "REPUBLICAN"')['candidatevotes'].sum() +
                            group[
                                (group['writein'] == False) &
                                (group['party_simplified'] != 'REPUBLICAN') &
                                (group['candidate'].isin(
                                    group.query('writein == False & party_simplified == "REPUBLICAN"')['candidate'].unique()
                                ))
                            ]['candidatevotes'].sum()
                        ),
                        'DEMOCRAT': (
                            group.query('writein == False & party_simplified == "DEMOCRAT"')['candidatevotes'].sum() +
                            group[
                                (group['writein'] == False) &
                                (group['party_simplified'] != 'DEMOCRAT') &
                                (group['candidate'].isin(
                                    group.query('writein == False & party_simplified == "DEMOCRAT"')['candidate'].unique()
                                ))
                            ]['candidatevotes'].sum()
                        ),
                        'LIBERTARIAN': (
                            group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidatevotes'].sum() +
                            group[
                                (group['writein'] == False) &
                                (group['party_simplified'] != 'LIBERTARIAN') &
                                (group['candidate'].isin(
                                    group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidate'].unique()
                                ))
                            ]['candidatevotes'].sum()
                        )
                    },
                    other_candidates := group[
                        (group['writein'] == False) &
                        (group['party_simplified'] == 'OTHER') &
                        (~group['candidate'].isin(
                            group.query('writein == False & party_simplified == "REPUBLICAN"')['candidate'].unique()
                        )) &
                        (~group['candidate'].isin(
                            group.query('writein == False & party_simplified == "DEMOCRAT"')['candidate'].unique()
                        )) &
                        (~group['candidate'].isin(
                            group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidate'].unique()
                        ))
                    ],
                    [votes_dict.update({f"OTHER_{row['candidate']}": row['candidatevotes']})
                     for _, row in other_candidates.iterrows()],
                    round(max(votes_dict.values()) / group['candidatevotes'].sum(), 4)
                )[-1])()
            ),

            'second_place_party': (
                (group.nlargest(2, 'candidatevotes')['party_simplified'].iloc[1] if len(group) > 1 else '')
                if group['state_po'].iloc[0] != 'NY'
                else (lambda: (
                    votes_dict := {
                        'REPUBLICAN': (
                            group.query('writein == False & party_simplified == "REPUBLICAN"')['candidatevotes'].sum() +
                            group[
                                (group['writein'] == False) &
                                (group['party_simplified'] != 'REPUBLICAN') &
                                (group['candidate'].isin(
                                    group.query('writein == False & party_simplified == "REPUBLICAN"')['candidate'].unique()
                                ))
                            ]['candidatevotes'].sum()
                        ),
                        'DEMOCRAT': (
                            group.query('writein == False & party_simplified == "DEMOCRAT"')['candidatevotes'].sum() +
                            group[
                                (group['writein'] == False) &
                                (group['party_simplified'] != 'DEMOCRAT') &
                                (group['candidate'].isin(
                                    group.query('writein == False & party_simplified == "DEMOCRAT"')['candidate'].unique()
                                ))
                            ]['candidatevotes'].sum()
                        ),
                        'LIBERTARIAN': (
                            group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidatevotes'].sum() +
                            group[
                                (group['writein'] == False) &
                                (group['party_simplified'] != 'LIBERTARIAN') &
                                (group['candidate'].isin(
                                    group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidate'].unique()
                                ))
                            ]['candidatevotes'].sum()
                        )
                    },
                    other_candidates := group[
                        (group['writein'] == False) &
                        (group['party_simplified'] == 'OTHER') &
                        (~group['candidate'].isin(
                            group.query('writein == False & party_simplified == "REPUBLICAN"')['candidate'].unique()
                        )) &
                        (~group['candidate'].isin(
                            group.query('writein == False & party_simplified == "DEMOCRAT"')['candidate'].unique()
                        )) &
                        (~group['candidate'].isin(
                            group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidate'].unique()
                        ))
                    ],
                    [votes_dict.update({f"OTHER_{row['candidate']}": row['candidatevotes']})
                     for _, row in other_candidates.iterrows()],
                    sorted_votes := sorted(votes_dict.items(), key=lambda x: x[1], reverse=True),
                    second_key := sorted_votes[1][0] if len(sorted_votes) > 1 else '',
                    'OTHER' if second_key.startswith('OTHER_') else second_key
                )[-1])()
            ),

            'second_place_vote_share': (
                round(
                    group.nlargest(2, 'candidatevotes')['candidatevotes'].iloc[1] /
                    group['candidatevotes'].sum() if len(group) > 1 else 0, 4
                )
                if group['state_po'].iloc[0] != 'NY'
                else (lambda: (
                    votes_dict := {
                        'REPUBLICAN': (
                            group.query('writein == False & party_simplified == "REPUBLICAN"')['candidatevotes'].sum() +
                            group[
                                (group['writein'] == False) &
                                (group['party_simplified'] != 'REPUBLICAN') &
                                (group['candidate'].isin(
                                    group.query('writein == False & party_simplified == "REPUBLICAN"')['candidate'].unique()
                                ))
                            ]['candidatevotes'].sum()
                        ),
                        'DEMOCRAT': (
                            group.query('writein == False & party_simplified == "DEMOCRAT"')['candidatevotes'].sum() +
                            group[
                                (group['writein'] == False) &
                                (group['party_simplified'] != 'DEMOCRAT') &
                                (group['candidate'].isin(
                                    group.query('writein == False & party_simplified == "DEMOCRAT"')['candidate'].unique()
                                ))
                            ]['candidatevotes'].sum()
                        ),
                        'LIBERTARIAN': (
                            group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidatevotes'].sum() +
                            group[
                                (group['writein'] == False) &
                                (group['party_simplified'] != 'LIBERTARIAN') &
                                (group['candidate'].isin(
                                    group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidate'].unique()
                                ))
                            ]['candidatevotes'].sum()
                        )
                    },
                    other_candidates := group[
                        (group['writein'] == False) &
                        (group['party_simplified'] == 'OTHER') &
                        (~group['candidate'].isin(
                            group.query('writein == False & party_simplified == "REPUBLICAN"')['candidate'].unique()
                        )) &
                        (~group['candidate'].isin(
                            group.query('writein == False & party_simplified == "DEMOCRAT"')['candidate'].unique()
                        )) &
                        (~group['candidate'].isin(
                            group.query('writein == False & party_simplified == "LIBERTARIAN"')['candidate'].unique()
                        ))
                    ],
                    [votes_dict.update({f"OTHER_{row['candidate']}": row['candidatevotes']})
                     for _, row in other_candidates.iterrows()],
                    sorted_votes := sorted(votes_dict.values(), reverse=True),
                    round(sorted_votes[1] / group['candidatevotes'].sum() if len(sorted_votes) > 1 else 0, 4)
                )[-1])()
            ),
        }), include_groups=False)
        .reset_index()
        .sort_values('state')
        [['year', 'state', 'state_po', 'first_place_party', 'first_place_vote_share',
          'second_place_party', 'second_place_vote_share']]
        .reset_index(drop=True)
    )

    return result

def analyze_multiple_elections(df, years=None):
    """
    여러 년도의 선거 데이터를 한번에 분석하는 함수

    Parameters:
    -----------
    df : pandas.DataFrame
        미국 선거 데이터
    years : list, optional
        분석할 연도 리스트. None이면 모든 가능한 연도를 분석

    Returns:
    --------
    pandas.DataFrame
        모든 연도의 분석 결과를 합친 데이터프레임
    """

    if years is None:
        # 데이터에 실제로 존재하는 연도 중 유효한 연도만 선택
        valid_years = list(range(1976, 2021, 4))
        existing_years = df['year'].unique()
        years = [year for year in valid_years if year in existing_years]

    # 각 연도별로 분석 실행
    results = []
    for year in years:
        try:
            result = analyze_election_data(df, year)
            results.append(result)
            print(f"{year}년 분석 완료: {len(result)}개 주/지역")
        except Exception as e:
            print(f"{year}년 분석 실패: {e}")

    # 모든 결과 합치기
    if results:
        combined_result = pd.concat(results, ignore_index=True)
        return combined_result
    else:
        return pd.DataFrame()

### 함수 사용법 안내
# result_1976 = analyze_election_data(df, 1976)
# result_multi = analyze_multiple_elections(df, [1976, 1980, 1984])
# result_all = analyze_multiple_elections(df)  # 모든 연도

# temp1

## csv

In [7]:
# 1976부터 2020까지 4년 간격으로 반복
years = list(range(1976, 2021, 4))

for year in years:
    try:
        print(f"{year}년 데이터 처리 중...")

        # 기존 함수 사용해서 분석
        result = analyze_vote_aggregation(df, year)

        # 파일명 생성 및 저장
        filename = f'temp1_president_{year}.csv'
        result.to_csv(filename, encoding='utf-8-sig', index=False)

        print(f"{year}년 완료: {filename} 저장됨 ({len(result)}개 주/지역)")

    except Exception as e:
        print(f"{year}년 처리 실패: {e}")

print("\n모든 연도 처리 완료!")

1976년 데이터 처리 중...
1976년 완료: temp1_president_1976.csv 저장됨 (51개 주/지역)
1980년 데이터 처리 중...
1980년 완료: temp1_president_1980.csv 저장됨 (51개 주/지역)
1984년 데이터 처리 중...
1984년 완료: temp1_president_1984.csv 저장됨 (51개 주/지역)
1988년 데이터 처리 중...
1988년 완료: temp1_president_1988.csv 저장됨 (51개 주/지역)
1992년 데이터 처리 중...
1992년 완료: temp1_president_1992.csv 저장됨 (51개 주/지역)
1996년 데이터 처리 중...
1996년 완료: temp1_president_1996.csv 저장됨 (51개 주/지역)
2000년 데이터 처리 중...
2000년 완료: temp1_president_2000.csv 저장됨 (51개 주/지역)
2004년 데이터 처리 중...
2004년 완료: temp1_president_2004.csv 저장됨 (51개 주/지역)
2008년 데이터 처리 중...
2008년 완료: temp1_president_2008.csv 저장됨 (51개 주/지역)
2012년 데이터 처리 중...
2012년 완료: temp1_president_2012.csv 저장됨 (51개 주/지역)
2016년 데이터 처리 중...
2016년 완료: temp1_president_2016.csv 저장됨 (51개 주/지역)
2020년 데이터 처리 중...
2020년 완료: temp1_president_2020.csv 저장됨 (51개 주/지역)

모든 연도 처리 완료!


# temp2

## csv

In [8]:
# 1976부터 2020까지 4년 간격으로 반복
years = list(range(1976, 2021, 4))

for year in years:
    try:
        print(f"{year}년 데이터 처리 중...")

        # 기존 함수 사용해서 분석
        result = analyze_election_data(df, year)

        # 파일명 생성 및 저장
        filename = f'temp2_president_{year}.csv'
        result.to_csv(filename, encoding='utf-8-sig', index=False)

        print(f"{year}년 완료: {filename} 저장됨 ({len(result)}개 주/지역)")

    except Exception as e:
        print(f"{year}년 처리 실패: {e}")

print("\n모든 연도 처리 완료!")

1976년 데이터 처리 중...
1976년 완료: temp2_president_1976.csv 저장됨 (51개 주/지역)
1980년 데이터 처리 중...
1980년 완료: temp2_president_1980.csv 저장됨 (51개 주/지역)
1984년 데이터 처리 중...
1984년 완료: temp2_president_1984.csv 저장됨 (51개 주/지역)
1988년 데이터 처리 중...
1988년 완료: temp2_president_1988.csv 저장됨 (51개 주/지역)
1992년 데이터 처리 중...
1992년 완료: temp2_president_1992.csv 저장됨 (51개 주/지역)
1996년 데이터 처리 중...
1996년 완료: temp2_president_1996.csv 저장됨 (51개 주/지역)
2000년 데이터 처리 중...
2000년 완료: temp2_president_2000.csv 저장됨 (51개 주/지역)
2004년 데이터 처리 중...
2004년 완료: temp2_president_2004.csv 저장됨 (51개 주/지역)
2008년 데이터 처리 중...
2008년 완료: temp2_president_2008.csv 저장됨 (51개 주/지역)
2012년 데이터 처리 중...
2012년 완료: temp2_president_2012.csv 저장됨 (51개 주/지역)
2016년 데이터 처리 중...
2016년 완료: temp2_president_2016.csv 저장됨 (51개 주/지역)
2020년 데이터 처리 중...
2020년 완료: temp2_president_2020.csv 저장됨 (51개 주/지역)

모든 연도 처리 완료!


# temp3

## csv

In [9]:
result_all1 = process_all_years_vote_aggregation(df)

1976년 분석 완료: 51개 주/지역
1980년 분석 완료: 51개 주/지역
1984년 분석 완료: 51개 주/지역
1988년 분석 완료: 51개 주/지역
1992년 분석 완료: 51개 주/지역
1996년 분석 완료: 51개 주/지역
2000년 분석 완료: 51개 주/지역
2004년 분석 완료: 51개 주/지역
2008년 분석 완료: 51개 주/지역
2012년 분석 완료: 51개 주/지역
2016년 분석 완료: 51개 주/지역
2020년 분석 완료: 51개 주/지역


In [10]:
result_all1.to_csv("temp3_president.csv", index=False, encoding="utf-8-sig")

# temp4

## csv

In [11]:
result_all2 = analyze_multiple_elections(df)

1976년 분석 완료: 51개 주/지역
1980년 분석 완료: 51개 주/지역
1984년 분석 완료: 51개 주/지역
1988년 분석 완료: 51개 주/지역
1992년 분석 완료: 51개 주/지역
1996년 분석 완료: 51개 주/지역
2000년 분석 완료: 51개 주/지역
2004년 분석 완료: 51개 주/지역
2008년 분석 완료: 51개 주/지역
2012년 분석 완료: 51개 주/지역
2016년 분석 완료: 51개 주/지역
2020년 분석 완료: 51개 주/지역


In [12]:
result_all2.to_csv("temp4_president.csv", index=False, encoding="utf-8-sig")

## EDA

In [13]:
result_all2.head()

,year,state,state_po,first_place_party,first_place_vote_share,second_place_party,second_place_vote_share
0,1976,ALABAMA,AL,DEMOCRAT,0.5573,REPUBLICAN,0.4261
1,1976,ALASKA,AK,REPUBLICAN,0.5790,DEMOCRAT,0.3565
2,1976,ARIZONA,AZ,REPUBLICAN,0.5637,DEMOCRAT,0.3980
3,1976,ARKANSAS,AR,DEMOCRAT,0.6496,REPUBLICAN,0.3490
4,1976,CALIFORNIA,CA,REPUBLICAN,0.4975,DEMOCRAT,0.4795


In [14]:
result_all2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   year                     612 non-null    int64  
 1   state                    612 non-null    object 
 2   state_po                 612 non-null    object 
 3   first_place_party        612 non-null    object 
 4   first_place_vote_share   612 non-null    float64
 5   second_place_party       612 non-null    object 
 6   second_place_vote_share  612 non-null    float64
dtypes: float64(2), int64(1), object(4)
memory usage: 33.6+ KB


In [15]:
result_all2['first_place_party'].value_counts()

,count
first_place_party,
REPUBLICAN,359
DEMOCRAT,253


In [16]:
result_all2['second_place_party'].value_counts()

,count
second_place_party,
DEMOCRAT,358
REPUBLICAN,252
OTHER,2


- https://en.wikipedia.org/wiki/1992_United_States_presidential_election_in_Maine
- https://en.wikipedia.org/wiki/1992_United_States_presidential_election_in_Utah

In [17]:
result_all2.loc[result_all2['second_place_party'] == 'OTHER']

,year,state,state_po,first_place_party,first_place_vote_share,second_place_party,second_place_vote_share
223,1992,MAINE,ME,DEMOCRAT,0.3877,OTHER,0.3044
248,1992,UTAH,UT,REPUBLICAN,0.4336,OTHER,0.2734


# Batch CSV Files to ZIP

In [18]:
import zipfile
import glob

# Find all CSV files in current directory
csv_files = glob.glob('*.csv')

# Create ZIP file
with zipfile.ZipFile('all_csv_files.zip', 'w') as zipf:
   for file in csv_files:
       zipf.write(file)
       print(f"Added: {file}")  # Show progress

print(f"Total {len(csv_files)} files compressed.")

Added: temp2_president_2008.csv
Added: temp2_president_1996.csv
Added: temp2_president_2016.csv
Added: temp1_president_1996.csv
Added: temp1_president_1988.csv
Added: temp2_president_1988.csv
Added: temp2_president_1976.csv
Added: temp1_president_2016.csv
Added: temp1_president_1980.csv
Added: temp2_president_1992.csv
Added: temp2_president_2020.csv
Added: temp1_president_2012.csv
Added: temp4_president.csv
Added: temp1_president_2020.csv
Added: temp2_president_1984.csv
Added: temp1_president_2000.csv
Added: temp1_president_1992.csv
Added: temp3_president.csv
Added: temp1_president_2008.csv
Added: temp1_president_1984.csv
Added: temp1_president_2004.csv
Added: temp2_president_2000.csv
Added: temp1_president_1976.csv
Added: temp2_president_2004.csv
Added: temp2_president_2012.csv
Added: temp2_president_1980.csv
Total 26 files compressed.
